# Demonstration/Testing of Loading Data from S3 Buckets

This notebook demonstrates loading data from the lake: **bronze** (raw Statcast dailies), **processed** (legacy by-player paths), **silver** (engineered `player_year_features.parquet`), and **gold** (model-ready `player_year_features_preprocessed.parquet` after silver→gold preprocessing, plus **KNN similar-neighbor** tables `player_year_similar_neighbors.parquet` after the gold player-similarity step runs).

## Imports

In [ ]:
import pandas as pd
import s3fs
from collections import defaultdict
import re
from datetime import date, timedelta

pd.set_option('display.max_columns', None)

## Loading Data from Bronze Data Bucket

### Statcast

In [ ]:
# Building our key
BUCKET = "diamond-dna"
prefix = "bronze/statcast"

# Date Selection
s_date = date(2025, 11, 1)

path = f"s3://{BUCKET}/{prefix}/year={s_date.year}/date={s_date.strftime('%Y-%m-%d')}/statcast_pitches.parquet"

# Use AWS creds to access S3 buckets
fs = s3fs.S3FileSystem()

# Tell pandas/pyarrow to use the S3 filesystem
df = pd.read_parquet(path, filesystem=fs)
df.head()

### Statcast Running

In [ ]:
# Building key
prefix = "bronze/statcast_running"

# Year selection
year = "2025"

path = f"s3://{BUCKET}/{prefix}/year={year}/statcast_sprint_speed.parquet"

# Use AWS creds to access S3 buckets
fs = s3fs.S3FileSystem()

# Tell pandas/pyarrow to use the S3 filesystem
df = pd.read_parquet(path, filesystem=fs)
df.head()

## Loading Data from Silver

### Pitcher

In [ ]:
# Building our key (legacy by-player pitch parquet path; current pipeline uses bronze dailies + silver feature tables)
BUCKET = "diamond-dna"
prefix = "silver"
role = 'pitcher'

# Year selection
year = '2025'

# Use AWS creds to access S3 buckets
fs = s3fs.S3FileSystem()

# Read as a single file stream so PyArrow doesn't discover Hive partitioning.
# (Partition discovery would add a dictionary-encoded "year" that conflicts with the int32 "year" column inside the file.)
path = f"s3://{BUCKET}/{prefix}/pitcher/year={year}/player_year_features.parquet"
fs = s3fs.S3FileSystem()

with fs.open(path, "rb") as f:
    df = pd.read_parquet(f)
    
df.head()

### Batter

In [ ]:
prefix = 'silver'
role = 'batter'

# Year selection
year = "2025"

# Use AWS creds to access S3 buckets
fs = s3fs.S3FileSystem()

path = f"s3://{BUCKET}/{prefix}/{role}/year={year}/player_year_features.parquet"
with fs.open(path, "rb") as f:
    df = pd.read_parquet(f)

pd.set_option('display.max_columns', None)
df.head()


## Loading gold (preprocessed) player-year feature tables

### Batter

In [ ]:
BUCKET = "diamond-dna"
prefix = "gold/statcast"  # default GOLD_PREFIX in src/pipeline/settings.py

role = "batter"
year = "2025"

fs = s3fs.S3FileSystem()
path = f"s3://{BUCKET}/{prefix}/{role}/year={year}/player_year_features_preprocessed.parquet"
with fs.open(path, "rb") as f:
    df_gold = pd.read_parquet(f)

pd.set_option("display.max_columns", None)
df_gold.head()

In [ ]:
import json

meta_path = f"s3://{BUCKET}/{prefix}/{role}/year={year}/preprocessing_metadata.json"
try:
    with fs.open(meta_path, "r") as f:
        meta = json.load(f)
    print("Top-level keys:", sorted(meta.keys()))
    print("Row count:", meta.get("row_count"))
    print("Feature columns:", len(meta.get("feature_columns", [])))
except FileNotFoundError:
    print(f"No metadata at {meta_path} (run silver→gold preprocessing for this role/year first).")

### Pitcher

In [ ]:
BUCKET = "diamond-dna"
prefix = "gold/statcast"

role = "pitcher"
year = "2025"

fs = s3fs.S3FileSystem()
path = f"s3://{BUCKET}/{prefix}/{role}/year={year}/player_year_features_preprocessed.parquet"
with fs.open(path, "rb") as f:
    df_gold = pd.read_parquet(f)

df_gold.head()

## Gold KNN similar neighbors

### Batter

In [ ]:
BUCKET = "diamond-dna"
prefix = "gold/statcast"  # default GOLD_PREFIX in src/pipeline/settings.py
role = "batter"
year = "2025"

fs = s3fs.S3FileSystem()
nbr_path = f"s3://{BUCKET}/{prefix}/{role}/year={year}/player_year_similar_neighbors.parquet"
try:
    with fs.open(nbr_path, "rb") as f:
        df_neighbors_batter = pd.read_parquet(f)
except FileNotFoundError:
    print(
        f"No neighbor table at {nbr_path} "
        "(run gold player similarity for this role/year after archetype clustering)."
    )
df_neighbors_batter.head()

### Pitcher

In [ ]:
BUCKET = "diamond-dna"
prefix = "gold/statcast"
role = "pitcher"
year = "2025"

fs = s3fs.S3FileSystem()
nbr_path = f"s3://{BUCKET}/{prefix}/{role}/year={year}/player_year_similar_neighbors.parquet"
try:
    with fs.open(nbr_path, "rb") as f:
        df_neighbors_pitcher = pd.read_parquet(f)
except FileNotFoundError:
    print(
        f"No neighbor table at {nbr_path} "
        "(run gold player similarity for this role/year after archetype clustering)."
    )
df_neighbors_pitcher.head()

### Similarity metadata (both roles)

In [ ]:
import json

BUCKET = "diamond-dna"
prefix = "gold/statcast"
year = "2025"
fs = s3fs.S3FileSystem()

for role in ("batter", "pitcher"):
    meta_path = f"s3://{BUCKET}/{prefix}/{role}/year={year}/player_similarity_metadata.json"
    try:
        with fs.open(meta_path, "r") as f:
            sim_meta = json.load(f)
        print(role, "—", "neighbor_table_rows:", sim_meta.get("neighbor_table_rows"), "| method:", sim_meta.get("similarity_method"))
    except FileNotFoundError:
        print(f"No similarity metadata at {meta_path}")